# 📓 Semana 20 · Dia 4 — Permissões por fluxo e perfis de acesso

**Curso**: Especialista Databricks — Engenharia de Dados → GenAI → Agentes

| Campo | Valor |
|---|---|
| **Plano** | ✅ Free Edition |
| **Tempo estimado** | 2h |
| **Certificação alvo** | Portfólio empresarial |
| **Pré-requisitos** | SQL e Python básicos · notebooks anteriores do curso |
| **Entregável do dia** | RBAC por fluxo aplicado |

---


## 📖 Teoria — RBAC por fluxo

Nem todo usuário edita todos os fluxos: o admin define **quem pode ver, editar ou aprovar** cada fluxo — armazenado em tabela de permissões e reforçado pelas permissões do UC.


### 💻 Na prática — Permissões por fluxo

Crie a tabela de permissões e as funções de checagem.


In [ ]:
%sql
CREATE TABLE IF NOT EXISTS workspace.app.permissoes_fluxo (
  fluxo_id STRING, usuario STRING, permissao STRING
) USING DELTA;
INSERT INTO workspace.app.permissoes_fluxo VALUES
  ('metas', 'ana', 'editar'),
  ('metas', 'joao', 'ver'),
  ('descontos', 'ana', 'aprovar');

In [ ]:
# Função de autorização (app)
def pode(usuario, fluxo_id, acao):
    r = spark.sql(f"""
      SELECT permissao FROM workspace.app.permissoes_fluxo
      WHERE fluxo_id = '{fluxo_id}' AND usuario = '{usuario}'
    """).collect()
    hierarquia = {"ver": 1, "editar": 2, "aprovar": 3}
    return any(hierarquia.get(p.permissao, 0) >= hierarquia[acao] for p in r)
print("ana edita metas:", pode("ana", "metas", "editar"))
print("joao edita metas:", pode("joao", "metas", "editar"))
print("ana aprova descontos:", pode("ana", "descontos", "aprovar"))

### 💻 Na prática — Integrando ao app

No Streamlit: o usuário logado (current_user) define as opções visíveis.


In [ ]:
# UI com RBAC
print("""
1. Obtenha o usuário (auth do workspace)
2. Liste só os fluxos onde pode(usuario, fluxo, "ver")
3. Editor YAML: só com permissão "editar"
4. Botão aprovar: só com "aprovar"
""")
print("RBAC no app + RBAC do UC (dynamic views) = dupla camada.")

> 🎯 **Dica de prova**: Portfólio: RBAC por fluxo (tabela de permissões) + UC (dynamic views) — o app respeita as duas camadas. Pergunta: 'como controlar acesso por fluxo?' → tabela de permissões + checagem no app.


## 🎯 Exercícios de fixação

**1.** Adicione o perfil 'admin' com acesso total.

**2.** Por que o UC (dynamic views) também protege os dados?

**3.** O que acontece se o usuário não tem permissão?


> Tente resolver **antes** de olhar o gabarito no final do notebook.


## 🗝️ Gabarito comentado

**1.** Admin

Crie regra: se usuario == admin → todas as ações em todos os fluxos.

**2.** UC protege

Mesmo que o app tenha bug, o UC limita os dados visíveis (defesa em profundidade).

**3.** Sem permissão

A UI esconde e a API bloqueia (403) — nunca confiar só na UI.



## ✅ Checklist de fechamento

- [ ] Rodei todas as células do notebook do início ao fim sem erros.
- [ ] Consigo explicar os conceitos de hoje em 3 frases (sem olhar o material).
- [ ] Fiz os exercícios e conferi o gabarito.
- [ ] Anotei as dúvidas que preciso revisar.

---
*Próximo passo: siga para o notebook seguinte do plano do curso.*